# Dataset 5 (iTIC traffic events) — EDA via DuckDB

**6,148,474 rows / ~4.8 GB across three CSVs. Never `pd.read_csv` these.**
DuckDB streams from disk and aggregates in SQL; only the small result ever reaches pandas.

| file | rows |
|---|---|
| `traffic_event_2021.csv` | 4,284,228 |
| `traffic_event_2022.csv` | 1,336,360 |
| `traffic_event_2023.csv` | 527,886 |

Schema: `eid`, `title_th`, `description`, `latitude`, `longitude`, `type`, `start_time`, `stop_time`.

### Purpose
Dataset 5 becomes **static per-hex-cell spatial priors**, never time-varying features — it ends in 2023 while the test period is 2024–25, so any time-keyed feature would be unavailable at inference and leak in training. This EDA answers the questions needed to design those priors.

### Questions
1. **Q1 — Distinct `type` values and counts.** Decides the per-type feature breakdown.
2. **Q2 — Are `eid`s duplicated across the three files?** Decides whether we dedup before counting.
3. **Q3 — Actual spatial bounding box.** Sizes the coverage and its gaps.
4. **Q4 — Event durations (`stop_time - start_time`).** Separates instantaneous incidents from multi-week closures.
5. **Q5 — Junk.** Null/zero coordinates, test rows, `stop_time < start_time`, absurd durations.

In [1]:
import sys

import duckdb

sys.path.insert(0, "..")
from src.training.config import ITIC_GLOB, TH_BBOX

con = duckdb.connect()

# GOTCHA (verified 2026-07-15, do not "simplify" this away):
# DuckDB infers start_time/stop_time as VARCHAR, not TIMESTAMP, because the columns mix
# two formats -- '2021-01-01 21:21:20' and the date-only '2023-01-05'. Every date function
# below therefore needs an explicit cast or it errors out. try_cast handles both formats:
# 0 rows out of 6,148,474 fail to cast.
#
# 1,228,423 rows (~20%!) have a date-only stop_time. Casting turns those into midnight,
# which DEFLATES their computed duration. That is a fifth of the data, not an edge case,
# so we carry an explicit stop_is_date_only flag rather than pretend the number is real.
con.execute(f"""
CREATE OR REPLACE VIEW itic AS
SELECT
    * EXCLUDE (start_time, stop_time),
    try_cast(start_time AS TIMESTAMP) AS start_time,
    try_cast(stop_time  AS TIMESTAMP) AS stop_time,
    length(trim(stop_time)) = 10      AS stop_is_date_only,
FROM read_csv(
    '{ITIC_GLOB}',
    header = true,
    filename = true,
    union_by_name = true,
    sample_size = -1          -- scan everything when inferring types; these files are messy
)
""")

# Expected: 2021 = 4,284,228 | 2022 = 1,336,360 | 2023 = 527,886 | total = 6,148,474
con.sql("SELECT filename[-8:] AS file, count(*) AS n FROM itic GROUP BY 1 ORDER BY 1")

┌──────────┬─────────┐
│   file   │    n    │
│ varchar  │  int64  │
├──────────┼─────────┤
│ 2021.csv │ 4284228 │
│ 2022.csv │ 1336360 │
│ 2023.csv │  527886 │
└──────────┴─────────┘

In [2]:
# Confirm the inferred schema before trusting anything below.
con.sql("DESCRIBE itic")

┌───────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name    │ column_type │  null   │   key   │ default │  extra  │
│      varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ eid               │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ title_th          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ description       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ latitude          │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ longitude         │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ type              │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ filename          │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ start_time        │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ stop_time         │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ stop_is_da

## Q1 — Distinct `type` values and counts
This directly decides the feature breakdown: which event categories get their own per-cell density column, and which are too rare to bother with (fold into `other`).

In [3]:
# Read the two count columns side by side.
#
# count(*) is ROW-weighted. Rows are periodic snapshots of *active* events, so a row count
# measures HOW LONG events stayed open. count(DISTINCT eid) is EVENT-weighted: it measures
# HOW MANY events happened. Different questions, opposite answers.
#
# Row-weighted, construction (งานก่อสร้าง) is the #1 category at 50.8%.
# Event-weighted, construction is 0.9% -- only 675 events, each re-emitted ~4,600 times
# because roadworks stay open for months -- and accidents (อุบัติเหตุ) are #1 at 58.0%.
#
# A density feature built on count(*) is a construction-duration proxy labelled as traffic
# risk. It looks plausible on a map and is wrong.
con.sql("""
WITH r AS (SELECT type, count(*)            AS n_rows   FROM itic GROUP BY 1),
     e AS (SELECT type, count(DISTINCT eid) AS n_events FROM itic GROUP BY 1)
SELECT
    e.type,
    r.n_rows,
    round(100.0 * r.n_rows   / sum(r.n_rows)   OVER (), 2) AS pct_rows,
    e.n_events,
    round(100.0 * e.n_events / sum(e.n_events) OVER (), 2) AS pct_events,
    round(1.0 * r.n_rows / e.n_events, 1)                  AS rows_per_event
FROM e JOIN r USING (type)
ORDER BY e.n_events DESC
""")

# TODO: settle the per-type feature breakdown off the n_events column, with someone who
# reads Thai. Do not guess what a category means.

┌─────────────────────┬─────────┬──────────┬──────────┬────────────┬────────────────┐
│        type         │ n_rows  │ pct_rows │ n_events │ pct_events │ rows_per_event │
│       varchar       │  int64  │  double  │  int64   │   double   │     double     │
├─────────────────────┼─────────┼──────────┼──────────┼────────────┼────────────────┤
│ อุบัติเหตุ              │  964342 │    15.68 │    45703 │      58.04 │           21.1 │
│ รถเสีย/กีดขวาง        │  180059 │     2.93 │    10532 │      13.37 │           17.1 │
│ น้ำท่วม                │  568890 │     9.25 │     7099 │       9.01 │           80.1 │
│ เพลิงไหม้             │  116743 │      1.9 │     5921 │       7.52 │           19.7 │
│ ปิดเบี่ยงการจราจร      │  168718 │     2.74 │     2941 │       3.73 │           57.4 │
│ รถติด/การจราจรหนาแน่น │   57804 │     0.94 │     2396 │       3.04 │           24.1 │
│ ฝนตก                │   40202 │     0.65 │     1984 │       2.52 │           20.3 │
│ งานก่อสร้าง           │ 3120923 │   

In [4]:
# Does the type mix shift year to year? If 2021 is dominated by a category that
# vanishes later, a prior pooled over 2021-2023 is really a prior about 2021.
con.sql("""
SELECT type, year(start_time) AS yr, count(*) AS n
FROM itic
GROUP BY 1, 2
ORDER BY type, yr
""")

┌─────────────────────┬───────┬─────────┐
│        type         │  yr   │    n    │
│       varchar       │ int64 │  int64  │
├─────────────────────┼───────┼─────────┤
│ การชุมนุม/เปลี่ยนเส้นทาง │  2020 │       1 │
│ การชุมนุม/เปลี่ยนเส้นทาง │  2021 │    4601 │
│ การชุมนุม/เปลี่ยนเส้นทาง │  2022 │    2029 │
│ การชุมนุม/เปลี่ยนเส้นทาง │  2023 │     139 │
│ งานก่อสร้าง           │  2015 │   14905 │
│ งานก่อสร้าง           │  2017 │  142844 │
│ งานก่อสร้าง           │  2018 │  627036 │
│ งานก่อสร้าง           │  2019 │  498527 │
│ งานก่อสร้าง           │  2020 │  821134 │
│ งานก่อสร้าง           │  2021 │ 1013465 │
│     ·               │    ·  │     ·   │
│     ·               │    ·  │     ·   │
│     ·               │    ·  │     ·   │
│ เบี่ยงการจราจร        │  2021 │  376283 │
│ เบี่ยงการจราจร        │  2022 │  308307 │
│ เบี่ยงการจราจร        │  2023 │  122857 │
│ เพลิงไหม้             │  2020 │       8 │
│ เพลิงไหม้             │  2021 │   48082 │
│ เพลิงไหม้             │  2022 │ 

## Q2 — Are `eid`s duplicated across files?

**Answered: `eid` is NOT unique.**

```
rows: 6,148,474   |   distinct eids: 78,690   |   distinct full rows: 2,255,590
```

98.7% of rows are repeats. These files are **periodic snapshots of currently-active events**, not an event log — an event is re-emitted on every poll for as long as it stays open. One `eid` appears **37,872 times**. And `distinct_full_rows` (2.26M) far exceeds `distinct eids` (78,690), so the repeats are **not byte-identical**: content drifts as the event is updated.

The real dataset is **78,690 events**, which is a small table. DuckDB is still needed to *read* 4.8 GB, but once deduped this fits in pandas easily.

**Open:** which snapshot of an `eid` is canonical — first row, last row, or max `stop_time`? The dedup rule changes every duration statistic, so settle it before building the prior.

In [5]:
con.sql("""
SELECT
    count(*)            AS total_rows,
    count(DISTINCT eid) AS distinct_eids,
    count(*) - count(DISTINCT eid) AS duplicate_rows
FROM itic
""")

┌────────────┬───────────────┬────────────────┐
│ total_rows │ distinct_eids │ duplicate_rows │
│   int64    │     int64     │     int64      │
├────────────┼───────────────┼────────────────┤
│    6148474 │         78690 │        6069784 │
└────────────┴───────────────┴────────────────┘

In [6]:
# If duplicates exist: are they identical rows, or the same eid with different content?
# Identical  -> safe to DISTINCT away.
# Conflicting -> eid is not a stable key; decide a tie-break rule and log it.
con.sql("""
SELECT eid,
       count(*)                          AS n_rows,
       count(DISTINCT filename)          AS n_files,
       count(DISTINCT (title_th, latitude, longitude, start_time)) AS n_variants
FROM itic
GROUP BY eid
HAVING count(*) > 1
ORDER BY n_rows DESC
LIMIT 20
""")

┌────────┬────────┬─────────┬────────────┐
│  eid   │ n_rows │ n_files │ n_variants │
│ int64  │ int64  │  int64  │   int64    │
├────────┼────────┼─────────┼────────────┤
│ 671633 │  37872 │       3 │          1 │
│ 721666 │  35666 │       3 │          1 │
│ 721669 │  35664 │       3 │          1 │
│ 689539 │  32038 │       2 │          1 │
│ 685712 │  30830 │       2 │          1 │
│ 752224 │  25826 │       2 │          1 │
│ 746618 │  23019 │       3 │          1 │
│ 740700 │  21726 │       3 │          1 │
│ 699031 │  17407 │       2 │          1 │
│ 767974 │  17342 │       2 │          1 │
│ 702181 │  13300 │       1 │          1 │
│ 629580 │  13076 │       1 │          1 │
│ 781096 │  12184 │       2 │          1 │
│ 781097 │  12184 │       2 │          1 │
│ 781105 │  12183 │       2 │          1 │
│ 781104 │  12183 │       2 │          1 │
│ 781102 │  12183 │       2 │          1 │
│ 781100 │  12183 │       2 │          1 │
│ 781099 │  12183 │       2 │          1 │
│ 781101 │ 

## Q3 — Actual spatial extent
The coverage claim is *Bangkok + vicinity only*. Verify it, and quantify how much of Thailand gets **no** iTIC signal — that is the population of cells that will carry `has_itic_coverage = false`.

Bangkok is roughly lat 13.5–14.0, lon 100.3–100.9, for reference.

In [8]:
con.sql("""
SELECT
    min(latitude)  AS min_lat, max(latitude)  AS max_lat,
    min(longitude) AS min_lon, max(longitude) AS max_lon,
    -- percentiles are the honest picture: min/max are one bad row away from lying
    quantile_cont(latitude,  [0.01, 0.25, 0.5, 0.75, 0.99]) AS lat_pctiles,
    quantile_cont(longitude, [0.01, 0.25, 0.5, 0.75, 0.99]) AS lon_pctiles
FROM itic
WHERE latitude IS NOT NULL AND longitude IS NOT NULL
""")

┌─────────────────┬───────────┬─────────────────┬─────────────────┬────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────┐
│     min_lat     │  max_lat  │     min_lon     │     max_lon     │                          lat_pctiles                           │                         lon_pctiles                          │
│     double      │  double   │     double      │     double      │                            double[]                            │                           double[]                           │
├─────────────────┼───────────┼─────────────────┼─────────────────┼────────────────────────────────────────────────────────────────┼──────────────────────────────────────────────────────────────┤
│ 5.7713827847745 │ 20.364776 │ 97.878496071344 │ 105.33302844247 │ [6.8111, 13.736466952381, 13.903377669415, 15.59931, 20.10124] │ [98.69413, 100.41015, 100.58114424348, 101.12856, 104.96586] │
└─────────────────┴─

In [ ]:
# How many events fall outside Thailand's bbox entirely? (Same junk class as dataset 1's 10 rows.)
con.sql(f"""
SELECT
    count(*) FILTER (WHERE latitude IS NULL OR longitude IS NULL)          AS null_coords,
    count(*) FILTER (WHERE latitude = 0 OR longitude = 0)                  AS zero_coords,
    count(*) FILTER (WHERE latitude  NOT BETWEEN {TH_BBOX["min_lat"]} AND {TH_BBOX["max_lat"]}
                        OR longitude NOT BETWEEN {TH_BBOX["min_lon"]} AND {TH_BBOX["max_lon"]}) AS outside_thailand,
    count(*)                                                               AS total
FROM itic
""")

## Q4 — Event durations
`stop_time - start_time`. A 40-minute accident and a 3-week construction closure are both one row, but they are not one unit of "risk evidence." This decides whether the prior counts events, sums durations, or does both.

**Known trap (already measured):** 1,228,423 rows — **~20% of the dataset** — have a date-only `stop_time` with no clock component. Cast to TIMESTAMP they become midnight, so their duration is deflated by up to 24 hours. The query below therefore splits on `stop_is_date_only` instead of pooling. Any duration statistic computed across both groups at once is wrong.

In [9]:
# Durations, SPLIT BY stop_is_date_only. Pooling them would be a lie: a date-only
# stop_time is cast to midnight, so its duration is deflated by up to 24h -- and 20%
# of rows are in that bucket. Read the two groups separately.
con.sql("""
WITH d AS (
    SELECT type,
           stop_is_date_only,
           date_diff('minute', start_time, stop_time) AS mins
    FROM itic
    WHERE start_time IS NOT NULL AND stop_time IS NOT NULL
)
SELECT
    type,
    stop_is_date_only,
    count(*)                                  AS n,
    count(*) FILTER (WHERE mins <  0)         AS negative_duration,   -- stop before start = junk
    count(*) FILTER (WHERE mins =  0)         AS zero_duration,
    round(median(mins), 1)                    AS median_mins,
    round(quantile_cont(mins, 0.90), 1)       AS p90_mins,
    round(quantile_cont(mins, 0.99), 1)       AS p99_mins,
    max(mins)                                 AS max_mins
FROM d
GROUP BY type, stop_is_date_only
ORDER BY n DESC
""")

# TODO: decide how the prior weights events -- plain count, duration-weighted,
# or both as separate features. If duration-weighted, the date-only rows need a rule
# (exclude them? treat as full-day?).

┌─────────────────────┬───────────────────┬─────────┬───────────────────┬───────────────┬─────────────┬───────────┬───────────┬──────────┐
│        type         │ stop_is_date_only │    n    │ negative_duration │ zero_duration │ median_mins │ p90_mins  │ p99_mins  │ max_mins │
│       varchar       │      boolean      │  int64  │       int64       │     int64     │   double    │  double   │  double   │  int64   │
├─────────────────────┼───────────────────┼─────────┼───────────────────┼───────────────┼─────────────┼───────────┼───────────┼──────────┤
│ งานก่อสร้าง           │ false             │ 3118948 │                 0 │             0 │    440186.5 │ 1576800.0 │ 2102400.0 │  3133012 │
│ อุบัติเหตุ              │ false             │  964241 │                 0 │           209 │        60.0 │      60.0 │     104.0 │    21660 │
│ เบี่ยงการจราจร        │ true              │  620596 │                 0 │          4705 │    131040.0 │  688320.0 │ 2568960.0 │  2568960 │
│ น้ำท่วม          

## Q5 — Junk hunt
Test rows, placeholder text, impossible timestamps. Anything found needs an explicit removal rule.

In [10]:
con.sql("""
SELECT
    count(*) FILTER (WHERE title_th   IS NULL OR trim(title_th)   = '') AS empty_title,
    count(*) FILTER (WHERE description IS NULL OR trim(description) = '') AS empty_desc,
    count(*) FILTER (WHERE start_time IS NULL)                          AS null_start,
    count(*) FILTER (WHERE stop_time  IS NULL)                          AS null_stop,
    count(*) FILTER (WHERE stop_time  <  start_time)                    AS stop_before_start,
    count(*) FILTER (WHERE lower(title_th) LIKE '%test%'
                        OR lower(description) LIKE '%test%')            AS looks_like_test
FROM itic
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────┬────────────┬───────────┬───────────────────┬─────────────────┐
│ empty_title │ empty_desc │ null_start │ null_stop │ stop_before_start │ looks_like_test │
│    int64    │   int64    │   int64    │   int64   │       int64       │      int64      │
├─────────────┼────────────┼────────────┼───────────┼───────────────────┼─────────────────┤
│           0 │         88 │          0 │         0 │             18619 │            5300 │
└─────────────┴────────────┴────────────┴───────────┴───────────────────┴─────────────────┘

In [11]:
# Eyeball a sample. This is where a human reading Thai catches what SQL cannot.
con.sql("SELECT eid, title_th, description, type, start_time, stop_time FROM itic USING SAMPLE 20 ROWS")

┌────────┬──────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────────┬─────────────────────┬─────────────────────┐
│  eid   │                       title_th                       │                                                                                                                                                                                                                                                            description                             

## Next: the static prior (Phase 1, after the questions above are answered)

Once `type` groups and the dedup rule are settled, aggregate to per-H3-cell density.
**Do not run this before the H3 resolution is decided** — it is written here to show the shape of the output, not to be run blind.

```sql
-- Sketch. h3_latlng_to_cell needs the DuckDB h3 community extension:
--   INSTALL h3 FROM community; LOAD h3;
-- Alternative: aggregate raw counts here, apply h3-py in pandas afterwards.
SELECT
    h3_latlng_to_cell(latitude, longitude, <RES>) AS h3_cell,
    type,
    count(DISTINCT eid)                            AS n_events,
    sum(date_diff('minute', start_time, stop_time)) AS total_event_minutes
FROM itic
WHERE <cleaning rules from Q5>
GROUP BY 1, 2;
```

Then pivot to one column per `type`, and join onto the accident hex grid with
**`has_itic_coverage`** — cells outside Bangkok get 0 density *and* `false`.
**Zero means unknown, not safe.** Without that flag the model reads the entire
rest of Thailand as the safest place in the country.